In [ ]:
import sys
print(sys.executable)

/Users/ahthini/Desktop/DissProject/myenv/bin/python


### **T1.2 – MIMIC-IV database structure study**

Output a list of columns, the shape and the head of each file to understand the data structure and identify relevant columns for our analysis 

In [4]:
import os
import pandas as pd
hosp_path = "/Users/ahthini/Desktop/DissProject/mimic-iv-3.1/hosp/"
icu_path = "/Users/ahthini/Desktop/DissProject/mimic-iv-3.1/icu/"
hospfiles = os.listdir(hosp_path)
icufiles = os.listdir(icu_path)

#hosp module output 
for file in hospfiles:
    if file.endswith(".csv.gz"):
        print("-------------------------------------------------------------------------------")
        print("FILE:", file)
        path = os.path.join(hosp_path, file)
        df = pd.read_csv(path, nrows=5)
        print("Columns")
        print(df.columns.tolist())
        print("Shape:", df.shape)
        print("Head:", df.head())

#icu module output
for file in icufiles:
    if file.endswith(".csv.gz"):
        print("-------------------------------------------------------------------------------")
        print("FILE:", file)
        path = os.path.join(icu_path, file)
        df = pd.read_csv(path, nrows=5)
        print("Columns")
        print(df.columns.tolist())
        print("Shape:", df.shape)
        print("Head:", df.head())

-------------------------------------------------------------------------------
FILE: poe.csv.gz
Columns
['poe_id', 'poe_seq', 'subject_id', 'hadm_id', 'ordertime', 'order_type', 'order_subtype', 'transaction_type', 'discontinue_of_poe_id', 'discontinued_by_poe_id', 'order_provider_id', 'order_status']
Shape: (5, 12)
Head:          poe_id  poe_seq  subject_id   hadm_id            ordertime  \
0  10000032-100      100    10000032  22841357  2180-06-26 22:09:02   
1  10000032-101      101    10000032  22841357  2180-06-26 22:09:02   
2  10000032-102      102    10000032  22841357  2180-06-26 22:09:02   
3  10000032-103      103    10000032  22841357  2180-06-26 22:09:02   
4  10000032-104      104    10000032  22841357  2180-06-26 22:09:02   

     order_type      order_subtype transaction_type  discontinue_of_poe_id  \
0   Medications                NaN              New                    NaN   
1  General Care        Code status              New                    NaN   
2     Nutritio

### **T1.3 – Define psychiatric cohort using ICD codes**

**Define file paths**

Psychiatric diagnoses are in the hosp module so only that is used for T1.3. The output folder created is where cohort files are saved. 

In [5]:
from pathlib import Path
#base directory where MIMIC-IV is stored
base_path = Path("/Users/ahthini/Desktop/DissProject/mimic-iv-3.1")

#path to hospital module (we only need hosp for diagnoses)
hosp_path = base_path / "hosp"

#folder to save outputs
output_path = Path("/Users/ahthini/Desktop/DissProject/outputs")
output_path.mkdir(parents=True, exist_ok=True)  #create folder if it doesn't exist
print("Output path:", output_path)


Output path: /Users/ahthini/Desktop/DissProject/outputs


**Load required tables**

diagnoses_icd contains diagnosis codes for each hospital admission
d_icd_diagnoses is the dictionary that explains what each diagnosis code means

In [6]:
import pandas as pd

#diagnoses_icd → contains diagnosis codes per admission
diagnoses_path = hosp_path / "diagnoses_icd.csv.gz"

#d_icd_diagnoses → dictionary to translate codes into human-readable text
icd_dict_path = hosp_path / "d_icd_diagnoses.csv.gz"

#load data
diagnoses = pd.read_csv(diagnoses_path)
icd_dict = pd.read_csv(icd_dict_path)

#quick check of dataset sizes
print("diagnoses shape:", diagnoses.shape)
print("icd dictionary shape:", icd_dict.shape)

diagnoses shape: (6364488, 5)
icd dictionary shape: (112107, 3)


**Clean ICD codes, extract ICD-9 numeric prefixes, and define psychiatric condditions**

Cleaning the International Classification of Diseases (ICD) codes make it easier to work with and allows for consistency. Here we convert codes to text, remove extra spaces, and make letters uppercase. Instead of writing the condition in full (depression, anxiety, schizophrenia), they store these conditions as codes (F32, F30, F41) for easier processing. 

icd_version tells us which version/system is being used. ICD-9 is the older system whereas ICD-10 is the newer system. ICD-9 is mostly numeric. In ICD-9, diagnoses are organised into number ranges (first three digits), where each range represents a category of diseases. 001-139 is infections, 140-239 is cancers, 240-279 is endocrine diseases, 280-289 is blood diseases, 290-319 is all mental and behavioural disorders etc. This is why we only preserve the number range 290-319 as this is relevant to what we want. In ICD-10, codes includes letters and numbers (F32) where the letter classifies condition type. F type codes are mental and behavioural disorder codes so these are retained to create a dataframe as well as the ICD-9 290-319 range.

The dataframe created only contains psychiatric diagnosis records. These codes mean nothing to humans, so we use d_icd_diagnoses which tells us the meaning of these codes, and use merge to attach this extra information to rows where both code and ICD version are the same. This adds a colum called long_title, giving more meaning to the new table.



In [7]:
#convert codes to string and standardise formatting - (important because some codes may be numeric or have spaces)
diagnoses["icd_code"] = diagnoses["icd_code"].astype(str).str.strip().str.upper()
icd_dict["icd_code"] = icd_dict["icd_code"].astype(str).str.strip().str.upper()

#ICD-9 codes are numeric - take first 3 digits and convert to numeric for range filtering (290–319)
diagnoses["icd_prefix_3"] = pd.to_numeric(
    diagnoses["icd_code"].str[:3],
    errors="coerce")  #if conversion fails, set to NaN

#ICD-9 psychiatric codes: 290–319
icd9_psych_mask = (
    (diagnoses["icd_version"] == 9) &
    (diagnoses["icd_prefix_3"].between(290, 319)))

#ICD-10 psychiatric codes: start with "F"
icd10_psych_mask = (
    (diagnoses["icd_version"] == 10) &
    (diagnoses["icd_code"].str.startswith("F")))

#keep only rows where diagnosis is psychiatric
psychiatric_diagnoses = diagnoses[icd9_psych_mask | icd10_psych_mask].copy()

#merge with dictionary to get long description of diagnosis
psychiatric_diagnoses = psychiatric_diagnoses.merge(
    icd_dict, on=["icd_code", "icd_version"], how="left") #left used so nothing gets removed despite NaN output

#basic checks
print("psychiatric diagnosis rows:", psychiatric_diagnoses.shape[0])
print("unique psychiatric patients:", psychiatric_diagnoses["subject_id"].nunique())
print("unique psychiatric admissions:", psychiatric_diagnoses["hadm_id"].nunique())

#show sample rows to verify results look correct
print("\nExample psychiatric diagnoses:")
print(psychiatric_diagnoses[
        ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version", "long_title"]
    ].head(20))

psychiatric diagnosis rows: 423435
unique psychiatric patients: 107956
unique psychiatric admissions: 238565

Example psychiatric diagnoses:
    subject_id   hadm_id  seq_num icd_code  icd_version  \
0     10000032  22595853        6    29680            9   
1     10000032  22595853        7    30981            9   
2     10000032  22841357        8     3051            9   
3     10000032  25742920        9     3051            9   
4     10000032  29079034        7     3051            9   
5     10000032  29079034       12    29680            9   
6     10000068  25022803        1    30500            9   
7     10000084  23052089        2    F0280           10   
8     10000084  29888819        3    F0280           10   
9     10000117  22927623        6     F419           10   
10    10000117  27988844       11     F419           10   
11    10000690  23280645       11     3004            9   
12    10000690  25860671       20     3004            9   
13    10000690  26146595        4

**Cohort creating and file save**

Keeps only one row per psychiatric admission. The project aim is to predict 30-day psychiatric readmission so first we need people who actually had a psychiatric admission which is what T1.3 does. 

In [8]:

#we only need unique admissions for modelling
psychiatric_admissions = psychiatric_diagnoses[
    ["subject_id", "hadm_id"]].drop_duplicates()

print("\npsychiatric admissions cohort shape:", psychiatric_admissions.shape)

#save full diagnosis-level dataset
psychiatric_diagnoses.to_csv(
    output_path / "psychiatric_diagnoses.csv",
    index=False)

#save admission-level cohort
psychiatric_admissions.to_csv(
    output_path / "psychiatric_admissions.csv",
    index=False)

print("\nSaved files:")
print(output_path / "psychiatric_diagnoses.csv")
print(output_path / "psychiatric_admissions.csv")


psychiatric admissions cohort shape: (238565, 2)

Saved files:
/Users/ahthini/Desktop/DissProject/outputs/psychiatric_diagnoses.csv
/Users/ahthini/Desktop/DissProject/outputs/psychiatric_admissions.csv


### **T1.4 - Define 30-day readmission outcome**

**Load admissions and psychiatric admissions**

The admissions table and psychiatric admissions tables are loaded and the relevant columns are kept. Readmission is defined based on time between discharge and next admission so we only need timing and IDs, nothing demographic necessary just yet. Psychiatric admissions are loaded as we need to know which admissions are psychiatric and whether the next admission is psychiatric.


In [9]:
#paths
base_path = "/Users/ahthini/Desktop/DissProject/mimic-iv-3.1/hosp/"
output_path = "/Users/ahthini/Desktop/DissProject/outputs/"

#load admissions
admissions = pd.read_csv(base_path + "admissions.csv.gz")
print("Admissions shape:", admissions.shape)

#keep only needed columns and convert to datetime
admissions = admissions[["subject_id", "hadm_id", "admittime", "dischtime"]]
admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])
print("Admissions shape after datetime conversion:", admissions.shape)

#load psychiatric admissions (from T1.3)
psych_adm = pd.read_csv(output_path + "psychiatric_admissions.csv")
print("Psychiatric admissions shape:", psych_adm.shape)

#mark psychiatric admissions, merge with admissions adnd fill missing as 0 (non-psychiatric)
psych_adm["is_psych"] = 1
adm = admissions.merge(psych_adm, on=["subject_id", "hadm_id"], how="left")
adm["is_psych"] = adm["is_psych"].fillna(0)
print("Merged admissions shape:", adm.shape)

Admissions shape: (546028, 16)
Admissions shape after datetime conversion: (546028, 4)
Psychiatric admissions shape: (238565, 2)
Merged admissions shape: (546028, 5)


**Sort and generate next admissions**

We merge full admissions with psychiatric admissions. Every admission is labelled either 0 (non-psychiatric) or 1 (psychiatric), which allows us to analyse all admission sequences and not just psychiatric ones. Then admissions are sorted by patient and time. This is because patients can have multiple admissions and we need them in chronological order. For each patient, next admit time and next is psych is created as we need to compare current discharge and next admission.

In [10]:
#sort admissions per patient
adm = adm.sort_values(["subject_id", "admittime"])
print("Admissions sorted by subject_id and admittime.")
print(adm.head(5))

#get next admission info
adm["next_admittime"] = adm.groupby("subject_id")["admittime"].shift(-1)
adm["next_is_psych"] = adm.groupby("subject_id")["is_psych"].shift(-1)
print("Next admission columns preview:")
print(adm[["subject_id", "hadm_id", "admittime", "next_admittime", "next_is_psych"]].head(5))

Admissions sorted by subject_id and admittime.
   subject_id   hadm_id           admittime           dischtime  is_psych
0    10000032  22595853 2180-05-06 22:23:00 2180-05-07 17:15:00       1.0
1    10000032  22841357 2180-06-26 18:27:00 2180-06-27 18:49:00       1.0
3    10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00       1.0
2    10000032  25742920 2180-08-05 23:44:00 2180-08-07 17:50:00       1.0
4    10000068  25022803 2160-03-03 23:16:00 2160-03-04 06:26:00       1.0
Next admission columns preview:
   subject_id   hadm_id           admittime      next_admittime  next_is_psych
0    10000032  22595853 2180-05-06 22:23:00 2180-06-26 18:27:00            1.0
1    10000032  22841357 2180-06-26 18:27:00 2180-07-23 12:35:00            1.0
3    10000032  29079034 2180-07-23 12:35:00 2180-08-05 23:44:00            1.0
2    10000032  25742920 2180-08-05 23:44:00                 NaT            NaN
4    10000068  25022803 2160-03-03 23:16:00                 NaT            NaN


**Time difference and defining readmission label**
The time difference is calculated as readmission depends on how long after discharge the patient returns. Negative gaps are for when next admissions appear before discharge as events such as transfers, overlapping records or admin timing can happen. 

In [11]:
#calculate time difference
adm["days_to_next"] = (adm["next_admittime"] - adm["dischtime"]).dt.days
#negative time gaps check
negative_gaps = adm[adm["days_to_next"] < 0]
print("Number of negative gaps:", len(negative_gaps))
print("Percentage of dataset:", (len(negative_gaps) / len(adm)) * 100)
print("\nExample negative gaps:")
print(negative_gaps[[
    "subject_id", 
    "hadm_id", 
    "admittime", 
    "dischtime", 
    "next_admittime", 
    "days_to_next"
]].head(5))
print("Time gap stats:")
print(adm["days_to_next"].describe())
print("Sample time gaps:")
print(adm[["subject_id", "hadm_id", "days_to_next"]].head(5))

#define readmission label
adm["readmitted_30d"] = (
    (adm["days_to_next"] <= 30) & 
    (adm["days_to_next"] >= 0) &
    (adm["next_is_psych"] == 1)
).astype(int)

print("Readmission label distribution:")
print(adm["readmitted_30d"].value_counts())

Number of negative gaps: 48
Percentage of dataset: 0.008790757983107094

Example negative gaps:
       subject_id   hadm_id           admittime           dischtime  \
3929     10076639  23684554 2135-12-22 22:53:00 2135-12-23 19:19:00   
25110    10482555  24141586 2146-10-02 00:11:00 2146-10-12 11:00:00   
36926    10693757  23954106 2163-04-20 00:00:00 2163-04-22 15:45:00   
53750    11011076  27826883 2178-12-02 21:43:00 2178-12-03 09:15:00   
71747    11339384  27383384 2187-09-16 04:36:00 2187-09-26 14:47:00   

           next_admittime  days_to_next  
3929  2135-12-23 17:19:00          -1.0  
25110 2146-10-02 20:16:00         -10.0  
36926 2163-04-20 15:46:00          -2.0  
53750 2178-12-03 07:15:00          -1.0  
71747 2187-09-26 09:30:00          -1.0  
Time gap stats:
count    322576.000000
mean        367.800531
std         655.022949
min         -31.000000
25%          16.000000
50%          83.000000
75%         402.000000
max        5340.000000
Name: days_to_next, dtype

**Cohort finalisation and file save**



In [12]:
#only keep psychiatric admissions as index cases
final_cohort = adm[adm["is_psych"] == 1]
print("Final cohort shape:", final_cohort.shape)
#save result and print stats
output_file = output_path + "psychiatric_readmission_cohort.csv"
final_cohort.to_csv(output_file, index=False)
print("Total psychiatric admissions:", len(final_cohort))
print("Readmitted within 30 days:", final_cohort["readmitted_30d"].sum())
print("Readmission rate (%):", final_cohort["readmitted_30d"].mean() * 100)
print("Saved to:", output_file)

Final cohort shape: (238565, 9)
Total psychiatric admissions: 238565
Readmitted within 30 days: 47367
Readmission rate (%): 19.854966151782534
Saved to: /Users/ahthini/Desktop/DissProject/outputs/psychiatric_readmission_cohort.csv


In [13]:
print(diagnoses[["icd_code","icd_version"]].head(50))
print(diagnoses["icd_version"].value_counts())
print(diagnoses.groupby("icd_version")["icd_code"].apply(lambda x: x.head(20).tolist()))

   icd_code  icd_version
0      5723            9
1     78959            9
2      5715            9
3     07070            9
4       496            9
5     29680            9
6     30981            9
7     V1582            9
8     07071            9
9     78959            9
10     2875            9
11     2761            9
12      496            9
13     5715            9
14      V08            9
15     3051            9
16    07054            9
17    78959            9
18     V462            9
19     5715            9
20     2767            9
21     2761            9
22      496            9
23      V08            9
24     3051            9
25    78791            9
26    45829            9
27    07044            9
28     7994            9
29     2761            9
30    78959            9
31     2767            9
32     3051            9
33      V08            9
34    V4986            9
35     V462            9
36      496            9
37    29680            9
38     5715            9
